# 03 — CLIP: Menghubungkan Gambar dan Teks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/05-vision-transformer/03_clip_zero_shot.ipynb)

Semua model sebelumnya hanya bisa mengenali kelas yang **ditetapkan saat training**. **CLIP** (*Contrastive Language–Image Pretraining*, OpenAI 2021) dilatih dengan 400 juta pasangan gambar–caption dari internet, sehingga ia memahami gambar **lewat bahasa**.

CLIP punya dua encoder:
- **Image encoder** (sebuah ViT) → vektor embedding gambar.
- **Text encoder** (sebuah Transformer) → vektor embedding teks.

Keduanya dilatih agar embedding gambar dan caption yang cocok **berdekatan** di ruang yang sama (*contrastive learning*). Hasilnya:
1. **Zero-shot classification**: klasifikasi ke kelas apa pun cukup dengan menulis nama kelasnya, tanpa training.
2. **Pencarian gambar ↔ teks**.
3. Embedding gambar berkualitas tinggi untuk **linear probe** dengan sedikit data.

Konsep embedding dan pencarian vektor ini adalah jembatan ke modul 07 (Embeddings & Vector DB).

> ⚖️ **Lisensi:** library `open_clip` berlisensi MIT; bobot CLIP OpenAI juga MIT.

## 0. Cek GPU

Di Colab pilih **Runtime → Change runtime type → T4 GPU**, lalu jalankan ulang dari atas. Notebook tetap bisa berjalan di CPU, tetapi jauh lebih lambat.

In [ ]:
import shutil
import subprocess
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU terdeteksi: {props.name} | memori {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU tidak terdeteksi, akan berjalan di CPU (lebih lambat).")
    print("   Di Colab: Runtime → Change runtime type → T4 GPU")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])

## 1. Instalasi dan Memuat Model

Kita memakai `open_clip`, implementasi open-source yang menyediakan bobot CLIP resmi OpenAI dan banyak varian lain. `create_model_and_transforms` mengembalikan model beserta transformasi gambar yang benar untuk model tersebut.

In [ ]:
%pip install -q open_clip_torch

Import, konfigurasi, dan memuat **ViT-B/32** dengan bobot OpenAI. `QUICK_RUN=1` memakai subset kecil.

In [ ]:
import os
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
import open_clip
from torch.utils.data import DataLoader, Subset
from torchvision import datasets

device = "cuda" if torch.cuda.is_available() else "cpu"
QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
N_TEST = 200 if QUICK_RUN else 2000        # jumlah gambar test yang dievaluasi
DATA_DIR = "./data"

model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
tokenizer = open_clip.get_tokenizer("ViT-B-32")
model = model.to(device).eval()

n_img = sum(p.numel() for p in model.visual.parameters())
n_all = sum(p.numel() for p in model.parameters())
print(f"device: {device} | parameter image encoder: {n_img / 1e6:.0f} juta | total: {n_all / 1e6:.0f} juta")
print("transformasi gambar:", preprocess)

## 2. Data: CIFAR-10

Kita memakai CIFAR-10 (modul 02) sebagai bahan uji. CLIP **tidak pernah dilatih** khusus untuk CIFAR-10, jadi ini murni uji kemampuan zero-shot. Transformasi `preprocess` dari CLIP memperbesar gambar 32×32 menjadi 224×224.

In [ ]:
test_full = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=preprocess)
CLASS_NAMES = test_full.classes
idx = torch.randperm(len(test_full), generator=torch.Generator().manual_seed(0))[:N_TEST].tolist()
test_set = Subset(test_full, idx)
test_loader = DataLoader(test_set, batch_size=100, num_workers=2)
raw_test = datasets.CIFAR10(DATA_DIR, train=False, download=True)   # gambar asli untuk ditampilkan
print("kelas:", CLASS_NAMES, "| jumlah uji:", len(test_set))

## 3. Embedding Gambar dan Teks

Langkah dasar CLIP:
1. `model.encode_image` → embedding gambar (512 dimensi untuk ViT-B/32).
2. `model.encode_text` → embedding teks, juga 512 dimensi, di **ruang yang sama**.
3. **Normalisasi L2** keduanya, sehingga dot product = **cosine similarity**.
4. Kalikan dengan `logit_scale` (≈ 100, suhu yang dipelajari) lalu softmax untuk mendapat probabilitas.

Kita hitung embedding semua gambar uji sekali saja, lalu memakainya berulang kali.

In [ ]:
@torch.no_grad()
def encode_images(loader):
    feats, labels = [], []
    for x, y in loader:
        f = model.encode_image(x.to(device))
        feats.append(F.normalize(f, dim=-1).float().cpu()); labels.append(y)
    return torch.cat(feats), torch.cat(labels)


@torch.no_grad()
def encode_texts(texts):
    f = model.encode_text(tokenizer(texts).to(device))
    return F.normalize(f, dim=-1).float().cpu()


img_feats, img_labels = encode_images(test_loader)
print("embedding gambar:", tuple(img_feats.shape), "| norma:", img_feats.norm(dim=-1)[:3])
print("logit_scale:", model.logit_scale.exp().item())

## 4. Zero-Shot Classification

Ubah setiap nama kelas menjadi kalimat (**prompt**), encode sebagai teks, lalu pilih kelas yang embedding teksnya paling mirip dengan embedding gambar. Tidak ada training sama sekali!

Kita bandingkan tiga strategi prompt:
1. **Nama kelas saja**: `"cat"`.
2. **Template**: `"a photo of a cat."`. Caption di internet jarang berupa satu kata, jadi kalimat utuh lebih mirip data training CLIP.
3. **Prompt ensembling**: rata-rata embedding dari beberapa template (`"a blurry photo of a cat."`, `"a low resolution photo of a cat."`, …). Cocok karena gambar CIFAR buram dan beresolusi rendah.

In [ ]:
TEMPLATES = [
    "a photo of a {}.", "a blurry photo of a {}.", "a low resolution photo of a {}.",
    "a photo of the small {}.", "a pixelated photo of a {}.", "a close-up photo of a {}.",
]


def zero_shot(class_texts):
    # class_texts: list berisi list kalimat per kelas; embedding kalimat dirata-rata per kelas
    bobot = []
    for kalimat in class_texts:
        e = encode_texts(kalimat).mean(0)
        bobot.append(F.normalize(e, dim=-1))
    W = torch.stack(bobot)                           # (n_kelas, 512)
    logits = model.logit_scale.exp().item() * img_feats @ W.T
    return (logits.argmax(1) == img_labels).float().mean().item(), logits


strategi = {
    "nama kelas saja": [[c] for c in CLASS_NAMES],
    "template 'a photo of a {}.'": [[f"a photo of a {c}."] for c in CLASS_NAMES],
    f"ensembling {len(TEMPLATES)} template": [[t.format(c) for t in TEMPLATES] for c in CLASS_NAMES],
}
hasil = {}
for nama, teks in strategi.items():
    acc, logits = zero_shot(teks)
    hasil[nama] = logits
    print(f"{nama:<30} akurasi zero-shot: {acc:.3f}")

Contoh prediksi zero-shot beserta 3 kelas teratas dan probabilitasnya:

In [ ]:
logits = hasil[f"ensembling {len(TEMPLATES)} template"]
probs = logits.softmax(dim=-1)

fig, axes = plt.subplots(2, 5, figsize=(15, 6.5))
for j, ax in enumerate(axes.flat):
    img, y = raw_test[idx[j]]
    top_p, top_k = probs[j].topk(3)
    ax.imshow(img)
    teks = "\n".join(f"{CLASS_NAMES[k]}: {p:.0%}" for p, k in zip(top_p, top_k))
    ax.set_title(f"asli: {CLASS_NAMES[y]}\n{teks}", fontsize=8, color="green" if top_k[0] == y else "red")
    ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Pencarian Teks → Gambar

Karena gambar dan teks berada di ruang yang sama, kita bisa **mencari gambar dengan kalimat bebas**, termasuk konsep yang bukan nama kelas CIFAR-10. Inilah dasar *semantic image search*; di modul 07 kita akan menyimpan embedding seperti ini di **vector database**.

In [ ]:
queries = ["a red car", "an animal in the water", "a flying object in the sky", "a horse on grass"]
q_feats = encode_texts(queries)
sim = q_feats @ img_feats.T                          # (n_query, n_gambar) cosine similarity

fig, axes = plt.subplots(len(queries), 5, figsize=(12, 2.6 * len(queries)))
for i, q in enumerate(queries):
    top_s, top_i = sim[i].topk(5)
    for j in range(5):
        img, y = raw_test[idx[top_i[j]]]
        axes[i, j].imshow(img)
        axes[i, j].set_title(f"{top_s[j]:.3f} ({CLASS_NAMES[y]})", fontsize=8)
        axes[i, j].axis("off")
    axes[i, 0].text(-40, 16, q, fontsize=10, ha="right", va="center")
plt.tight_layout(); plt.show()

## 6. Linear Probe: Sedikit Data Berlabel

Embedding gambar CLIP juga merupakan **fitur** yang sangat baik. Dengan melatih klasifier linear sederhana (logistic regression) di atas embedding tersebut, cukup beberapa contoh per kelas sudah bisa mengalahkan zero-shot.

Kita ambil *k* gambar berlabel per kelas dari data **train**, latih logistic regression di embedding-nya, lalu uji di gambar test yang sama.

In [ ]:
from sklearn.linear_model import LogisticRegression

train_full = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=preprocess)
targets = np.array(train_full.targets)
rng = np.random.default_rng(0)

acc_zero_shot = (hasil[f"ensembling {len(TEMPLATES)} template"].argmax(1) == img_labels).float().mean().item()
print(f"zero-shot (0 contoh berlabel): {acc_zero_shot:.3f}")

for k in ([1, 4] if QUICK_RUN else [1, 4, 16, 64]):
    pilih = np.concatenate([rng.choice(np.where(targets == c)[0], k, replace=False) for c in range(10)])
    tr_feats, tr_labels = encode_images(DataLoader(Subset(train_full, pilih.tolist()), batch_size=100))
    clf = LogisticRegression(max_iter=2000, C=10.0).fit(tr_feats.numpy(), tr_labels.numpy())
    acc = (clf.predict(img_feats.numpy()) == img_labels.numpy()).mean()
    print(f"linear probe {k:>2} contoh/kelas ({10 * k:>3} gambar): {acc:.3f}")

## Ringkasan

- CLIP memetakan gambar dan teks ke **ruang embedding yang sama** lewat *contrastive learning* pada ratusan juta pasangan gambar–caption.
- **Zero-shot**: kelas cukup didefinisikan dengan teks; **prompt engineering** dan **ensembling** berpengaruh nyata pada akurasi.
- Embedding yang sama bisa dipakai untuk **pencarian teks → gambar** (fondasi *semantic search* dan vector DB di modul 07).
- **Linear probe** di atas embedding CLIP dengan sedikit data berlabel sering melampaui zero-shot.

## Latihan

1. **Prompt engineering.** Rancang minimal 3 set template baru (misalnya gaya lukisan, sudut pandang, atau bahasa Indonesia: `"foto seekor {}"`) dan ukur akurasi zero-shot masing-masing. Kelas mana yang paling diuntungkan atau dirugikan? Mengapa template berbahasa Indonesia kemungkinan lebih buruk untuk model ini?
2. **Klasifikasi gambar sendiri.** Upload 10 foto milikmu dan tentukan 5 kelas buatanmu sendiri (misalnya jenis makanan Indonesia: rendang, sate, nasi goreng, bakso, gado-gado). Jalankan zero-shot dan analisis kesalahannya. Coba perbaiki dengan deskripsi yang lebih kaya, misalnya `"a photo of rendang, a type of Indonesian food"`.
3. **Mesin pencari gambar mini.** Hitung embedding untuk 5.000 gambar CIFAR-10 test, simpan ke file `.npy`, lalu tulis fungsi `cari(teks, k=8)` yang menampilkan k gambar paling mirip. Ukur waktu pencariannya, lalu tuliskan ide: apa yang perlu diubah jika koleksinya berisi 10 juta gambar? (Ini persiapan untuk modul 07.)